In [1]:
import json
import numpy as np
import pandas as pd
import torch
import shap
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSequenceClassification

LABEL_COLS = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
PROC_DIR = Path("../data/processed")
MODELS_DIR = Path("../models")

metrics = json.loads(Path("../reports/metrics.json").read_text())
best_model_name = "bert" if metrics["bert"]["macro_f1"] >= metrics["roberta"]["macro_f1"] else "roberta"
print("Using best model for SHAP:", best_model_name, "- macro F1:", metrics[best_model_name]["macro_f1"])

model_dir = str(MODELS_DIR / best_model_name)


Using best model for SHAP: bert - macro F1: 0.5551000880363325


In [2]:
tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSequenceClassification.from_pretrained(model_dir)
model.eval()

def predict_fn(texts):
    enc = tokenizer(list(texts), truncation=True, max_length=128, padding=True, return_tensors="pt")
    with torch.no_grad():
        logits = model(**enc).logits
    return torch.sigmoid(logits).numpy()


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [3]:
test = pd.read_csv(PROC_DIR / "test_official_filtered.csv")

toxic_sample = test[test["toxic"] == 1].sample(10, random_state=42)
clean_sample = test[test["toxic"] == 0].sample(10, random_state=42)
sample = pd.concat([toxic_sample, clean_sample]).reset_index(drop=True)



sample["word_count"] = sample["comment_text"].str.split().str.len()
sample = sample[sample["word_count"] <= 40].head(10).reset_index(drop=True)
sample[["comment_text", "toxic", "insult", "identity_hate"]]


,comment_text,toxic,insult,identity_hate
0,":::::::Hey - Grow up, knock off the ridiculou...",1,1,0
1,== wow == \n\n thanks for locking the page... ...,1,1,0
2,HE IS SO FREAKIN SEXIIIIIIIIIIIIIIIIIIIIIIIIII...,1,0,0
3,"No Homo, his tities look kinda bigg, it looked...",1,0,0
4,Darren sucks so does erdal,1,1,0
5,""" \n :Haha, I'm so stupid... Sorry about that,...",1,0,0
6,Don't touch my aricle!!!! You sucks!!!,1,1,0
7,if you come to detroit you will get killed so...,1,0,0
8,") \n :: Formatting is very nice, it requires l...",0,0,0
9,Copyright violations \n Do not upload others' ...,0,0,0


In [4]:
masker = shap.maskers.Text(tokenizer)
explainer = shap.Explainer(predict_fn, masker, output_names=LABEL_COLS)

shap_values = explainer(sample["comment_text"].tolist(), max_evals=300)
print("SHAP values shape:", shap_values.shape)


PartitionExplainer explainer:  40%|████      | 4/10 [00:00<?, ?it/s]

PartitionExplainer explainer:  60%|██████    | 6/10 [00:10<00:00,  4.47it/s]

PartitionExplainer explainer:  70%|███████   | 7/10 [00:15<00:05,  1.85s/it]

PartitionExplainer explainer:  80%|████████  | 8/10 [00:17<00:04,  2.05s/it]

PartitionExplainer explainer:  90%|█████████ | 9/10 [00:20<00:02,  2.22s/it]

PartitionExplainer explainer: 100%|██████████| 10/10 [00:23<00:00,  2.59s/it]

PartitionExplainer explainer: 11it [00:26,  2.62s/it]                        

PartitionExplainer explainer: 11it [00:26,  3.72s/it]

SHAP values shape: (10, None, 6)


In [5]:
shap.plots.text(shap_values[0, :, "toxic"])


In [6]:
shap.plots.text(shap_values[5, :, "toxic"])


In [7]:

clean_idx = sample[sample["toxic"] == 0].index[0]
shap.plots.text(shap_values[clean_idx, :, "toxic"])


In [8]:
shap.plots.text(shap_values[0, :, "identity_hate"])
